In [2]:
import pandas as pd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

In [3]:
import pandas as pd

file_path = "/kaggle/input/flight-delay-dataset-20182022/Combined_Flights_2018.csv"

df = pd.read_csv(file_path)

df.info()


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/flight-delay-dataset-20182022/Combined_Flights_2018.csv'

In [ ]:
df.head()

In [ ]:
missing = df.isnull().sum()
missing[missing > 1]


In [ ]:
cols_to_drop = [
    "DepDelay",
    "ArrDelayMinutes",
    "ArrDelay",
    "Tail_Number",
    "DepDel15",
    "DepartureDelayGroups",
    "TaxiOut",
    "WheelsOff",
    "WheelsOn",
    "TaxiIn",
    "ArrDel15",
    "ArrivalDelayGroups",
    'DOT_ID_Marketing_Airline',
    'IATA_Code_Marketing_Airline',
    'Flight_Number_Marketing_Airline',
    'DOT_ID_Operating_Airline',
    'IATA_Code_Operating_Airline',
    'Flight_Number_Operating_Airline',
    'OriginCityMarketID',
    'DestCityMarketID',
    'OriginAirportSeqID',
    'DestAirportSeqID',
    'OriginStateFips',
    'DestStateFips',
    'OriginWac',
    'DestWac',
    'Year'
]
d_C=df.copy()
d_C = d_C.drop(columns=cols_to_drop)


In [ ]:
# DepTime
print("DepTime missing:", d_C["DepTime"].isna().sum())

# ArrTime
print("ArrTime missing:", d_C["ArrTime"].isna().sum())

# AirTime
print("AirTime missing:", d_C["AirTime"].isna().sum())

# ActualElapsedTime
print("ActualElapsedTime missing:", d_C["ActualElapsedTime"].isna().sum())

# CRSElapsedTime
print("CRSElapsedTime missing:", d_C["CRSElapsedTime"].isna().sum())


In [ ]:
# حذف الصفوف اللي فيها NaN في DepTime أو ArrTime
d_C = d_C[d_C["DepTime"].notna() & d_C["ArrTime"].notna()]

# التأكد من عدد الصفوف بعد الحذف
d_C.shape


In [ ]:
d_C.info()

In [ ]:
d_C.duplicated().sum()

In [ ]:
d_C["FlightDate"] = pd.to_datetime(d_C["FlightDate"])


In [ ]:
# Calculate actual flight delay in minutes
d_C["Delay"] = d_C["ActualElapsedTime"] - d_C["CRSElapsedTime"]

# 2️ Convert any negative values to zero
d_C["Delay"] = d_C["Delay"].clip(lower=0)

# 3️ Create a binary column for delayed flights
d_C["Delayed"] = (d_C["Delay"] >= 15).astype(int)

# 4️ Display counts of delayed vs on-time flights
print(d_C["Delayed"].value_counts())

# 5️ Show first 5 rows to verify
d_C.head()



In [ ]:
# ==========================
# 1️⃣ Convert departure times to minutes
# ==========================
# Convert actual departure time from HHMM to total minutes
d_C["Dep_mins"] = (d_C["DepTime"] // 100) * 60 + (d_C["DepTime"] % 100)
# Convert scheduled departure time from HHMM to total minutes
d_C["CRSDep_mins"] = (d_C["CRSDepTime"] // 100) * 60 + (d_C["CRSDepTime"] % 100)

# Calculate actual departure delay in minutes
d_C["DepDelay"] = d_C["Dep_mins"] - d_C["CRSDep_mins"]
# Replace negative values with 0 (departed earlier than scheduled)
d_C["DepDelay"] = d_C["DepDelay"].apply(lambda x: x if x > 0 else 0)

# ==========================
# 2️⃣ Convert arrival times to minutes
# ==========================
# Convert actual arrival time from HHMM to total minutes
d_C["Arr_mins"] = (d_C["ArrTime"] // 100) * 60 + (d_C["ArrTime"] % 100)
# Convert scheduled arrival time from HHMM to total minutes
d_C["CRSArr_mins"] = (d_C["CRSArrTime"] // 100) * 60 + (d_C["CRSArrTime"] % 100)

# Calculate actual arrival delay in minutes
d_C["ArrDelay"] = d_C["Arr_mins"] - d_C["CRSArr_mins"]
# Replace negative values with 0 (arrived earlier than scheduled)
d_C["ArrDelay"] = d_C["ArrDelay"].apply(lambda x: x if x > 0 else 0)


Cancelled = True → flight was cancelled.

Diverted = True → flight landed at a different airport than planned.

value_counts() gives you the total number of True/False for each column.

In [ ]:
# ==========================
# 1️⃣ Count the number of cancelled flights
# ==========================
# 'Cancelled' is a boolean column: True if the flight was cancelled, False otherwise
print(d_C["Cancelled"].value_counts())

# ==========================
# 2️⃣ Count the number of diverted flights
# ==========================
# 'Diverted' is a boolean column: True if the flight was diverted, False otherwise
print(d_C["Diverted"].value_counts())


Delay = ActualElapsedTime - CRSElapsedTime (actual flight time minus scheduled flight time).

All negative delays have been set to 0, so now:

Delay = 0 → Flight arrived on-time or earlier than scheduled.

Delay > 0 → Flight was delayed (took longer than scheduled).

The describe() output shows:

count → Total number of flights.

mean → Average delay in minutes.

std → Standard deviation of delays.

min → Minimum delay (0 after adjustment).

25%, 50%, 75% → Quartiles of delay distribution.

max → Maximum delay observed.

In [ ]:
d_C["Delay"].describe()  # يعطيك إحصائيات: متوسط، أقصى، أدنى


In [ ]:
# متوسط التأخير لكل شركة تشغيل
d_C.groupby("Operating_Airline")["Delay"].mean().sort_values(ascending=False)


In [ ]:

d_C["Delayed"].value_counts().plot(
    kind="barh",
    figsize=(7,4),
    color=["green", "red"],
    title="Binary Flight Delay Classification"
)
plt.xlabel("Number of Flights")
plt.ylabel("Delay Class")
plt.show()


"Scatter plot showing the relationship between departure delays and arrival delays. Most flights cluster around zero delay, while extreme outliers indicate data errors or unusually long/short flights."

In [ ]:
plt.scatter(d_C['DepDelay'], d_C['ArrDelay'], alpha=0.5)
plt.title('Departure Delay vs Arrival Delay')
plt.xlabel('Departure Delay (minutes)')
plt.ylabel('Arrival Delay (minutes)')
plt.grid(True)
plt.show()
d_C[['DepDelay','ArrDelay']].corr()

**This plot shows the number of flights in 2018 categorized by delay severity,On Time/Early, Small, Medium,Large Delays, and Cancelled,highlighting that most flights departed on time or with only minimal delays.**

In [ ]:
pal = sns.color_palette()
d_C["DelayGroup"] = None
d_C.loc[df["DepDelayMinutes"] == 0, "DelayGroup"] = "OnTime_Early"
d_C.loc[
    (d_C["DepDelayMinutes"] > 0) & (d_C["DepDelayMinutes"] <= 15), "DelayGroup"
] = "Small_Delay"
d_C.loc[
    (d_C["DepDelayMinutes"] > 15) & (d_C["DepDelayMinutes"] <= 45), "DelayGroup"
] = "Medium_Delay"
d_C.loc[d_C["DepDelayMinutes"] > 45, "DelayGroup"] = "Large_Delay"
d_C.loc[d_C["Cancelled"], "DelayGroup"] = "Cancelled"

d_C["DelayGroup"].value_counts(ascending=True).plot(
    kind="barh", figsize=(10, 5), color='purple', title="Flight Results (2018)"
)
plt.show()

**This plot shows how flight frequency varies across the days of the week, revealing that weekdays generally have more flights than weekends.**

In [ ]:
# Count the occurrences of flights for each day of the week
day_of_week_counts = d_C['DayOfWeek'].value_counts().sort_index()

# Plotting a bar chart to visualize the distribution of flights across days of the week
plt.bar(day_of_week_counts.index, day_of_week_counts.values,color='purple')
plt.title('Distribution of Flights Across Days of the Week')
plt.xlabel('Day of the Week')
plt.ylabel('Number of Flights')
plt.xticks(range(1, 8), ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'], rotation=60)
plt.show()
df['DayOfWeek'].value_counts().sort_index()

The chart shows the distribution of flight delays: most flights arrive on time or with minor delays, while a very small number of flights experience long delays.

In [ ]:

sns.histplot(d_C["Delay"], bins=50, kde=True)
plt.title("Distribution of Flight Delays")
plt.xlabel("Delay (minutes)")
plt.ylabel("Number of Flights")
plt.show()


KS has the highest delay rate, with about 15% of its flights delayed.

Airlines like VX, C5, and AS have delay rates around 10%.

Most other airlines have delay rates below 10%, with EM and 9K having the lowest, under 3%.

In [ ]:
delayed_ratio = d_C.groupby("Operating_Airline")["Delayed"].mean().sort_values(ascending=False)
delayed_ratio.plot(kind="bar", figsize=(12,6), color="orange")
plt.ylabel("Percentage of Delayed Flights")
plt.title("Percentage of Delayed Flights per Airline")
plt.show()


The airport YNG has the highest delay rate, with more than 50% of its flights delayed — meaning about half of all flights experience delays, which is significant.

Next is CYS, with a delay rate of around 24%.

The remaining airports have delay rates ranging between 14% and 20%.

In [ ]:
top_origins = d_C.groupby("Origin")["Delayed"].mean().sort_values(ascending=False).head(10)
top_origins.plot(kind="bar", figsize=(12,6), color="green")
plt.ylabel("Percentage of Delayed Flights")
plt.title("Top 10 Origin Airports by Delay Rate")
plt.show()


This chart shows the top 10 flight routes with the highest delay rates.
Each bar represents a route from one airport to another with a specific airline.
The darker the color, the higher the delay percentage.
The top route has the most delays, and the rest are shown in descending order.

In [ ]:

top_routes = d_C.groupby(["Operating_Airline", "Origin", "Dest"])["Delayed"].mean().sort_values(ascending=False).head(10)

top_routes_df = top_routes.reset_index()
top_routes_df["Route"] = top_routes_df["Origin"] + " → " + top_routes_df["Dest"] + " (" + top_routes_df["Operating_Airline"] + ")"

plt.figure(figsize=(12,6))
sns.barplot(x="Delayed", y="Route", data=top_routes_df, palette="Reds_r")
plt.xlabel("Percentage of Delayed Flights")
plt.ylabel("Route")
plt.title("Top 10 Flight Routes by Delay Rate")
plt.xlim(0,1)
plt.show()


**Extra**

**Which distance group has the most flights?**

In [ ]:
sns.countplot(data=d_C,x='DistanceGroup')

**Which distance group of flights experiences the highest delay rate? Does the distance of the flight seem to affect the likelihood of being delayed?**

In [ ]:
import matplotlib.cm as cm

delayed_counts = d_C[d_C['Delayed']==True].groupby('DistanceGroup')['Delayed'].count()

total_counts = d_C.groupby('DistanceGroup')['Delayed'].count()

delay_rate = delayed_counts / total_counts

colors = cm.viridis(delay_rate / delay_rate.max())

plt.figure(figsize=(12,6))
bars = plt.bar(delay_rate.index, delay_rate, color=colors)
plt.title('Delay Rate by Distance Group', fontsize=14)
plt.xlabel('Distance Group', fontsize=12)
plt.ylabel('Delay Rate', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)

**How does the number of flights vary throughout the year? Which months have the highest and lowest flight activity?**

In [ ]:

monthly_counts = d_C.groupby('Month').size()

plt.figure(figsize=(12,6))
plt.plot(monthly_counts.index, monthly_counts.values, marker='o', linestyle='-', color='blue', linewidth=2)

plt.xticks(monthly_counts.index)
plt.xlabel("Month", fontsize=12)
plt.ylabel("Number of Flights", fontsize=12)
plt.title("Number of Flights per Month", fontsize=14)
plt.grid(True, linestyle='--', alpha=0.5)



plt.show()


**Which quarter had the highest number of diverted flights? What might be the reason for these diversions?**

In [ ]:

d_C[d_C['Diverted']==True].groupby('Quarter')['Diverted'].value_counts().sort_values(ascending=False).plot(kind='bar')

In [ ]:
d_C[d_C['Diverted']==True].groupby('Airline')['Diverted'].count().sort_values(ascending=False).head(10).plot(kind='bar')
plt.ylabel("Number of Diverted Flights")
plt.xlabel("Airline")
plt.title("Top 10 Airlines by Diverted Flights")
plt.show()


In [ ]:
sns.scatterplot(data=d_C, x='AirTime', y='ArrDelay', hue='DistanceGroup')
plt.title("Arrival Delay vs AirTime")
plt.show()


In [ ]:
numeric_df = d_C.select_dtypes(include=[np.number])
plt.figure(figsize=(20,10))
sns.heatmap(numeric_df.corr(),annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()


In [ ]:
d_C['Cancelled'] = d_C['Cancelled'].astype(int)
d_C['Diverted'] = d_C['Diverted'].astype(int)

print(d_C[['Cancelled', 'Diverted']].head())


In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder


In [ ]:
# Show only columns with missing values
missing = d_C.isnull().sum()
missing = missing[missing > 0]
print(missing)


**If you want to use all data without losing any, missing values in numeric columns will be filled using imputation.**

In [ ]:
numeric_cols = d_C.select_dtypes(include=['int64', 'float64']).columns.tolist()

if 'Delayed' in numeric_cols:
    numeric_cols.remove('Delayed')


X = d_C[numeric_cols]
y = d_C['Delayed']


**Imputation**

In [ ]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='median')

d_C[numeric_cols] = imputer.fit_transform(d_C[numeric_cols])

X = d_C[numeric_cols]
y = d_C['Delayed']

**If you prefer not to, columns containing missing values will be excluded before training.**

In [ ]:
numeric_cols = d_C.select_dtypes(include=['int64', 'float64']).columns.tolist()

if 'Delayed' in numeric_cols:
    numeric_cols.remove('Delayed')

cols_to_exclude = ['AirTime', 'CRSElapsedTime', 'ActualElapsedTime', 'Delay']
numeric_cols = [col for col in numeric_cols if col not in cols_to_exclude]

X = d_C[numeric_cols]
y = d_C['Delayed']


In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


**to check if we need Scaling or nah**

In [ ]:
X.describe()


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


**to check about imbalance**

In [ ]:
print(y.value_counts())
print(y.value_counts(normalize=True))


In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("Before SMOTE:\n", y_train.value_counts(normalize=True))
print("After SMOTE:\n", y_train_res.value_counts(normalize=True))
